# Trade-off table, blocking, and the first move toward optimization

**Source worksheets:** [yint.org/w8](https://yint.org/w8) and [yint.org/w9](https://yint.org/w9) - weeks 8 and 9 of the applied DoE short course.

Module 5 ended with the *defining relation* and the *resolution* of a
fractional design.  This module zooms out to the **trade-off table**
- the standard reference for picking ``k`` factors and ``p`` generators
when budget is tight - then works through a small case study that
combines a fractional factorial with **blocking** on the experimenter,
and finishes by introducing the **sequential experimentation** mindset
that drives Modules 7 and 8.

## Q1-Q8 - reading the trade-off table

A few quick lookups:

- **Five factors, sixteen runs**: $k = 5$, $p = 1$.  One generator;
  conventionally $E = ABCD$.  Defining relation $I = ABCDE$.
  Resolution V.
- **Six factors, twenty runs**: 20 is not a power of 2.  The nearest
  workable designs are 16 or 32 runs.  At $k = 6$, $p = 2$
  (so $2^{6-2} = 16$) the standard generators are $E = ABC$,
  $F = BCD$; the defining relation has $2^{2} - 1 = 3$ words.

The general pattern:

> Each generator costs one half of the runs.
> $p$ generators $\rightarrow$ $2^{k-p}$ runs $\rightarrow$
> $2^{p} - 1$ words in the defining relation $\rightarrow$
> $2^{p} - 1$ alias chains.

In Module 5 we saw $p = 1$ (half-fraction, one word $ABCD$).
With $p = 2$ (quarter-fraction) the defining relation has *three*
words: the two generators and their product.

In [ ]:
# 6 factors, 16 runs (k=6, p=2), generators E=ABC and F=BCD

generators = ["ABCE", "BCDF"]

# defining relation = I = ABCE = BCDF = ABCE * BCDF = (BC)^2 ADEF = ADEF

print(f"Generators (as words):   {generators}")
print(f"Defining relation: I = {' = '.join(generators)} = ADEF (= ABCE * BCDF, after cancellation)")

## Q9-Q13 - half-fraction in four factors, plus blocking on the experimenter

The set-up: four factors **A**, **B**, **C**, **D** to study; only 8
runs in the budget; the work must be split between you and a colleague
because the schedule does not allow one person to do all 8.

A natural design is a **half-fraction** with generator ``D = ABC``
(resolution IV).  Splitting the work between two experimenters is a
**blocking** problem: the experimenter is a **nuisance factor** - we
do not care whether the value is "you" or "colleague", but if we do
not control for it any drift in technique gets blamed on the real
factors.  Block on it by treating "Person" as an extra column ``E``
whose generator is some interaction we are willing to lose.

The standard choice is ``E = AB`` (block confounded with the AB
two-factor interaction).  Two design "words" gives us a resolution-III
design overall: every main effect is now confounded with a *two-factor*
interaction.

Worksheet response values (Q13):
``y = [120, 76, 106, 90, 72, 74, 90, 55]`` in standard order.

In [ ]:
from process_improve.experiments import c, gather, lm

A = c(-1, +1, -1, +1, -1, +1, -1, +1, name="A")
B = c(-1, -1, +1, +1, -1, -1, +1, +1, name="B")
C = c(-1, -1, -1, -1, +1, +1, +1, +1, name="C")

# Generator D = A*B*C

D = c(-1, +1, +1, -1, +1, -1, -1, +1, name="D")

# Block factor E = "Person", with E = A*B

E = c(+1, -1, -1, +1, +1, -1, -1, +1, name="E")
y = c(120, 76, 106, 90, 72, 74, 90, 55, name="y")
design = gather(A=A, B=B, C=C, D=D, E=E, y=y)
design

In [ ]:
# Fit main effects plus the one two-factor interaction (A:C) that is
# not already absorbed by a generator.

m = lm("y ~ A + B + C + D + E + A:C", design)
print(m.get_parameters(drop_intercept=False).to_string())

## Sequential experimentation - the bridge to optimization (w9)

A response surface study is not one big design; it is a *sequence* of
small designs, each pointing the next one in the right direction.  The
loop:

1. **Predict** the outcome of the next experiment with the current model.
2. **Run** the experiment.
3. **Compare** the prediction with the measurement.

   - If they agree, the model is still useful.  Decide:

     - extend the model with the new point and keep climbing, or
     - stop, because you are at the optimum.

   - If they disagree, the model has broken down.  Switch to a
     higher-order model (add a center point, then axial points for
     a quadratic) or shift the design region.

4. **Plan** where the next experiment will be.
5. **Repeat**.

The mantra: *the model is useful, but wrong* - useful because it gives
you a defensible direction, wrong because the true surface curves and
the linear approximation will break eventually.  Knowing when it
breaks is the entire skill.

## A worked 1-D optimization

To make the response-surface loop concrete, we will optimize a 1-D
process by hand.  Pick a single factor (think: temperature, mixer
speed, or any other physical knob), take small steps, fit a linear
model, swap to a quadratic when the surface starts to curve, predict
the peak, and confirm.

The "true" system below is hidden from the optimizer; only the
function call ``observe(t)`` is allowed - it returns the measured
response at temperature ``t`` with a sprinkle of measurement noise.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

rng = np.random.default_rng(seed=42)


def observe(temperature_C: float) -> float:
    """Return a noisy measurement: true peak at 60 degC, sigma = 1."""
    true = -0.05 * (temperature_C - 60) ** 2 + 75.0
    return float(true + rng.normal(scale=1.0))


# First three experiments: anchor a linear trend.

xs = [40.0, 50.0, 60.0]
ys = [observe(t) for t in xs]
for t, y in zip(xs, ys, strict=True):
    print(f"  t = {t:5.1f} degC -> measured y = {y:.2f}")

In [ ]:
# Fit a simple linear model on the first three points - is the surface
# still going up?

slope, intercept = np.polyfit(xs, ys, deg=1)
print(f"Linear fit (3 points): y = {intercept:.2f} + {slope:.3f} * t")
print("Slope is positive -> step further uphill.")

# Step uphill by +10 degC.

next_t = max(xs) + 10
xs.append(next_t)
ys.append(observe(next_t))
print(f"\nNext run at t = {next_t} degC -> y = {ys[-1]:.2f}")

In [ ]:
# Add a couple more steps; switch to a quadratic fit once we have 5
# points so curvature can show up.

for next_t in [70.0, 80.0]:
    xs.append(next_t)
    ys.append(observe(next_t))

coefs = np.polyfit(xs, ys, deg=2)
poly = np.poly1d(coefs)
predicted_peak = -coefs[1] / (2 * coefs[0])
print(f"Quadratic fit: y = {coefs[0]:.4f}*t^2 + {coefs[1]:.3f}*t + {coefs[2]:.2f}")
print(f"Predicted peak temperature: {predicted_peak:.1f} degC")
print(f"Predicted response at peak: {poly(predicted_peak):.2f}")

In [ ]:
# Confirm the predicted optimum with a fresh run, then plot the journey.

xs.append(predicted_peak)
ys.append(observe(predicted_peak))
print(f"Confirmation run at t = {predicted_peak:.1f} degC -> y = {ys[-1]:.2f}")

grid = np.linspace(35, 85, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=xs, y=ys, mode="markers+text",
                         text=[f"{i+1}" for i in range(len(xs))],
                         textposition="top center",
                         name="Observations", marker={"size": 9}))
fig.add_trace(go.Scatter(x=grid, y=poly(grid), mode="lines",
                         name="Quadratic fit",
                         line={"dash": "dash"}))
fig.update_layout(width=720, height=420,
                  title="1-D sequential optimization",
                  xaxis_title="Temperature [degC]",
                  yaxis_title="Response y")
fig

## Wrap-up

Three transferable points from this module:

- **The trade-off table** sets the budget conversation.  Pick the
  resolution you can afford, name the generators, write the defining
  relation, and *then* run the experiment.
- **Blocking** is how you keep nuisance factors honest.  Confound
  them with the highest-order interaction you are willing to lose.
- **Optimization is sequential.**  No single design lands you on
  the peak; a string of small designs does.

**Next:** Module 7 takes the same loop into **two factors** with the
classical response-surface trio of factorial + center + axial points.